# SpikeQuest: SNN Grid-World Navigation with R-STDP

This notebook walks through a complete experiment: training a spiking neural network (SNN) agent to navigate a grid world using **reward-modulated STDP** (three-factor learning rule) with novelty-driven exploration.

## Learning Rule Summary

**Three-factor R-STDP with eligibility traces:**

$$
\begin{align}
\tau_{\text{pre}}\frac{d}{dt}\bar{x} &= -\bar{x} + S_{\text{pre}}(t) \\
\tau_{\text{post}}\frac{d}{dt}\bar{y} &= -\bar{y} + S_{\text{post}}(t) \\
\tau_e\frac{d}{dt}e &= -e + \bar{x}(t)S_{\text{post}}(t) - \bar{y}(t)S_{\text{pre}}(t) \\
\Delta w_{ij} &= \eta \cdot M(t) \cdot e_{ij}(t)
\end{align}
$$

Where $M(t)$ is the neuromodulator (reward + novelty - baseline).

**Novelty bonus:** visited-state count or dual-timescale suppression.

## References
- Frémaux & Gerstner (2016). Reward-modulated STDP. *Front. Syn. Neurosci.*
- Izhikevich (2007). Solving the distal reward problem through linkage of STDP and dopamine signaling. *Cerebral Cortex.*
- Pathak et al. (2017). Curiosity-driven exploration by self-supervised prediction. *ICML.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from spikequest.env.grid_world import GridWorld
from spikequest.agents.spike_agent import SpikeAgent, train_episode
from spikequest.utils.seeding import set_seed

set_seed(42)
print("Imports OK")

In [ ]:
env = GridWorld(size=10, max_steps=200)
print(env.render_grid())
print(f"Observation dim: {env.get_obs_dim()}")

In [ ]:
agent = SpikeAgent(
    n_input=2 * env.size,
    n_hidden=64,
    n_output=4,
    T=10,
    novelty_coeff=1.0,
    rstdp_config=dict(tau_pre=20.0, tau_post=20.0, tau_elig=50.0,
                       lr=0.005, w_init=0.3, w_init_std=0.1),
    novelty_mode='visited',
)
print(f"Agent created. Network: {agent.policy.layer_sizes}")

In [ ]:
n_episodes = 100
successes, rewards, steps = [], [], []

for ep in range(n_episodes):
    r = train_episode(env, agent, max_steps=200)
    successes.append(r['success'])
    rewards.append(r['total_reward'])
    steps.append(r['steps'])
    
print(f"Final success rate: {np.mean(successes[-20:]):.2f}")
print(f"Avg steps (last 20): {np.mean(steps[-20:]):.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
w = 10
def smooth(x): return np.convolve(x, np.ones(w)/w, mode='valid')

axes[0].plot(smooth(successes)); axes[0].set_title('Success Rate')
axes[1].plot(smooth(rewards)); axes[1].set_title('Cumulative Reward')
axes[2].plot(smooth(steps)); axes[2].set_title('Steps')
for ax in axes: ax.set_xlabel('Episode')
plt.tight_layout()
plt.savefig('experiments/outputs/notebook_curves.png', dpi=150)
plt.show()

## Interpreting Results

- **Success rate** should increase over episodes as R-STDP reinforces action sequences leading to the goal.
- **Steps** should decrease as the agent learns more direct paths.
- **Cumulative reward** should increase with more efficient navigation.

### Limitations
- R-STDP is a local plasticity rule and may converge slower than backprop-based methods.
- The visited novelty bonus is heuristic; more sophisticated intrinsic motivation (prediction-error, count-based) may improve exploration.
- For larger grids, the population-code input encoding becomes inefficient.